# TennisMyLife — Gallica 1903 Colab worker — A100

Select **A100 GPU + High RAM**, then run the cells from top to bottom. When asked for a file, select **`tml_colab_ed25519`** from your Windows Downloads folder.

ALTO download and RapidOCR GPU run as two parallel branches. Every completed page is uploaded immediately to the VPS.


In [ ]:
!rm -rf /content/Tennis-OCR-Pipeline
!git clone -q https://github.com/Tennismylife/Tennis-OCR-Pipeline.git /content/Tennis-OCR-Pipeline
!pip -q uninstall -y onnxruntime onnxruntime-gpu >/dev/null 2>&1 || true
!pip -q install -r /content/Tennis-OCR-Pipeline/colab/requirements.txt


In [ ]:
import subprocess, onnxruntime as ort
subprocess.run(['nvidia-smi','--query-gpu=name,memory.total,driver_version','--format=csv,noheader'], check=True)
providers = ort.get_available_providers()
print('ONNX Runtime providers:', providers)
if 'CUDAExecutionProvider' not in providers:
    raise RuntimeError('CUDAExecutionProvider not available. Verify A100 runtime, then Runtime > Restart session and Run all again.')
print('GPU CHECK OK — RapidOCR will use CUDA')


In [ ]:
from google.colab import files
import base64
uploaded = files.upload()
if not uploaded:
    raise RuntimeError('No SSH key uploaded')
key_name, key_bytes = next(iter(uploaded.items()))
if key_name.endswith('.pub'):
    raise RuntimeError('Upload the private key tml_colab_ed25519, not the .pub file')
KEY_B64 = base64.b64encode(key_bytes).decode()
print('SSH key loaded:', key_name)


In [ ]:
import sys
sys.path.insert(0, '/content/Tennis-OCR-Pipeline/colab')
from worker import connect_sftp

VPS_HOST = 'vibrant-lovelace.82-165-11-122.plesk.page'
VPS_USER = 'andre'
VPS_PORT = 2222
VPS_MANIFEST = '/home/andre/GallicaJobs/gallica-1903-all-tennis/GALlica_1903_ALL_TENNIS/00_MANIFEST/colab_active_claims.tsv'
VPS_REMOTE_CACHE = '/home/andre/GallicaJobs/gallica-1903-all-tennis/GALlica_1903_ALL_TENNIS/ocr_cache_latin_full/targeted_remaining_1903'
VPS_ALTO_CACHE = '/home/andre/GallicaJobs/_shared/alto_cache'

tr, sftp = connect_sftp(VPS_HOST, VPS_USER, KEY_B64, VPS_PORT)
sftp.get(VPS_MANIFEST, '/content/colab_claim.tsv')
sftp.close(); tr.close()
rows = sum(1 for _ in open('/content/colab_claim.tsv', encoding='utf-8-sig')) - 1
print('Claim downloaded. Rows:', rows)


In [ ]:
import csv
src='/content/colab_claim.tsv'
with open(src,encoding='utf-8-sig',newline='') as f:
    rr=list(csv.DictReader(f,delimiter='\t'))
fields=list(rr[0]) if rr else ['ark','page','mode']
for mode,path in [('ALTO','/content/colab_alto.tsv'),('RAPID','/content/colab_rapid.tsv')]:
    subset=[r for r in rr if (r.get('mode') or '').upper()==mode]
    with open(path,'w',encoding='utf-8-sig',newline='') as f:
        w=csv.DictWriter(f,fieldnames=fields,delimiter='\t'); w.writeheader(); w.writerows(subset)
    print(mode, 'rows:', len(subset))


In [ ]:
import subprocess, os
base = [
    'python', '/content/Tennis-OCR-Pipeline/colab/worker.py',
    '--vps-host', VPS_HOST, '--vps-user', VPS_USER, '--vps-key-b64', KEY_B64,
    '--vps-port', str(VPS_PORT), '--remote-cache', VPS_REMOTE_CACHE,
    '--remote-alto-cache', VPS_ALTO_CACHE, '--profile', 'HQ', '--max-pages', '0'
]
alto_cmd = base + ['--manifest','/content/colab_alto.tsv','--device','cpu','--delay','15']
rapid_cmd = base + ['--manifest','/content/colab_rapid.tsv','--device','cuda','--delay','0']
print('Starting ALTO and A100 RapidOCR in parallel...')
p_alto = subprocess.Popen(alto_cmd)
p_rapid = subprocess.Popen(rapid_cmd)
rapid_rc = p_rapid.wait()
alto_rc = p_alto.wait()
print('Branches complete: ALTO rc=', alto_rc, 'RapidOCR/A100 rc=', rapid_rc)
if alto_rc != 0 or rapid_rc != 0:
    raise RuntimeError(f'Branch failure ALTO={alto_rc} RAPID={rapid_rc}')


If the Colab runtime stops, reopen this notebook and run all cells again. Pages already uploaded to the VPS are detected and skipped. The GPU verification cell prevents accidental CPU fallback.
